# Data Splits: Random vs Time-Based

The EDA's central warning was temporal leakage with 90% of ratings in 2000, a random split lets the model train on a user's future and test on their past. This notebook makes that concrete with two splits built from the same raw ratings.

## Setup

In [ ]:
import sys
from pathlib import Path

# set root directory
ROOT = Path.cwd()
while not (ROOT / "recommendation_lab").is_dir():
    ROOT = ROOT.parent
    if ROOT == ROOT.parent:
        raise RuntimeError("could not locate recommendation_lab package")
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from recommendation_lab.data.loader import load_ml_1m
from recommendation_lab.data.split import random_split, time_base_split

# load dataset
ratings = load_ml_1m()["ratings"]
print(f"raw ratings: {len(ratings):,}")

Dataset 'ml-1m' already present at /Users/kayceejenz/Documents/experiment/recommendation-system-lab/data/ml-1m
raw ratings: 1,000,209


## The two splits

- `random_split`: each user's ratings are shuffled (seeded) and split proportionally, 80/20.
- `time_base_split`: each user's ratings are sorted by timestamp; the latest 20% become test.

Both guarantee every user keeps training data, and both drop test rows whose movie never appears in train (cold items), which we report explicitly.

In [ ]:
# split data
rs = random_split(ratings, seed=42)
ts = time_base_split(ratings)

summary = pd.DataFrame({
    "random": [len(rs.train), len(rs.test)],
    "time-based": [len(ts.train), len(ts.test)],
}, index=["train rows", "test rows"])

print(summary.to_string())
print()
print(f"users in train: random {rs.train['user_id'].nunique()} / time {ts.train['user_id'].nunique()} "
      f"(total {ratings['user_id'].nunique()})")

            random  time-based
train rows  797758      797758
test rows   202433      202397

users in train: random 6040 / time 6040 (total 6040)


## Quantifying leakage

For every user, we count test rows whose rating happened *before* one of their own training ratings. In a time-based split this is impossible by construction. In a random split it happens for roughly half of all test rows: the model is being asked to 'predict' the past, which it has already seen.

In [3]:
def leakage_fraction(split):
    merged = split.test.merge(split.train, on="user_id", suffixes=("_test", "_train"))
    return (merged["timestamp_test"] < merged["timestamp_train"]).mean()

print(f"random split   leakage: {leakage_fraction(rs):.1%} of test rows")
print(f"time-based split leakage: {leakage_fraction(ts):.1%} of test rows")

random split   leakage: 49.7% of test rows
time-based split leakage: 0.0% of test rows


## Cold-item filtering

Test ratings for movies the training set never saw cannot be scored by neighborhood or factorization models. Both splitters remove them by default; this quantifies how much of the test set that is.

In [4]:
for name, split in [("random", rs), ("time-based", ts)]:
    test_items = set(split.test["movie_id"])
    train_items = set(split.train["movie_id"])
    cold = test_items - train_items
    print(f"{name:10s} test movies: {len(test_items):,} | cold (unseen in train): {len(cold):,}")

unfiltered = time_base_split(ratings, filter_cold=False)
print(f"\ntime-based test rows before filter: {len(unfiltered.test):,} | after: {len(ts.test):,} "
      f"({len(unfiltered.test) - len(ts.test):,} dropped)")

random     test movies: 3,435 | cold (unseen in train): 0
time-based test movies: 3,496 | cold (unseen in train): 0

time-based test rows before filter: 202,451 | after: 202,397 (54 dropped)


## Determinism

Both splits are reproducible: the time-based split needs no randomness at all, and the random split is seeded.

In [5]:
rs2 = random_split(ratings, seed=42)
ts2 = time_base_split(ratings)
print("random split  deterministic:", rs.train.equals(rs2.train) and rs.test.equals(rs2.test))
print("time-based split deterministic:", ts.train.equals(ts2.train) and ts.test.equals(ts2.test))

random split  deterministic: True
time-based split deterministic: True
